# 08 — Carteras 2025
Usa el mejor modelo (ventana de salida = 90 días) para construir dos carteras sobre datos de 2025:

- **Cartera 1 — Buy & Hold**: pesos iguales entre los 23 activos.
- **Cartera 2 — NN**: long activos con retorno predicho positivo, short negativos.

Pesos **fijos** durante todo 2025 (calculados con datos hasta 2024).
`[EXTENDER]` Rebalanceo mensual: recalcular pesos cada ~21 días de trading.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings, yfinance as yf
warnings.simplefilter('ignore')

from keras import Sequential, Input
from keras.layers import LSTM, Dense

from utils import (TICKERS, create_time_series_data, make_splits,
                   eval_mae, get_callbacks, compile_model)

## 1. Entrenamiento del mejor modelo (V_out=90)
Entrenar con todos los datos históricos hasta 2024 incluido.
Reemplazar `build_best` por la arquitectura ganadora de `06_resultados.ipynb`.

In [ ]:
V_IN  = 10    # ventana de entrada del mejor modelo — ajustar según resultados
V_OUT = 90    # ventana de salida fija para carteras
EPOCHS = 100
BATCH_SIZE = 64

# Datos históricos hasta 2024
precios_hist = yf.download(TICKERS, start='1945-01-01', end='2025-01-01',
                           auto_adjust=True, progress=False)['Close']
precios_hist.dropna(axis=1, inplace=True)
returns_hist = np.log(precios_hist).diff().dropna()
print(f'Datos históricos: {returns_hist.shape}')

In [ ]:
# Modelo de referencia — reemplazar por el ganador real
def build_best(V_in):
    return compile_model(Sequential([
        Input((V_in, 23)), LSTM(64), Dense(23)]))

X, y = create_time_series_data(returns_hist, V_IN, V_OUT)
X_tr, X_v, X_ts, y_tr, y_v, y_ts = make_splits(X, y)

model = build_best(V_IN)
hist = model.fit(X_tr, y_tr, validation_data=(X_v, y_v),
                 epochs=EPOCHS, batch_size=BATCH_SIZE,
                 callbacks=get_callbacks(), verbose=1)

print(f'\nMAE train={eval_mae(model,X_tr,y_tr):.4f}  '
      f'val={eval_mae(model,X_v,y_v):.4f}  '
      f'test={eval_mae(model,X_ts,y_ts):.4f}')

# Curva de convergencia
plt.figure(figsize=(7, 3))
plt.plot(hist.history['loss'], label='train')
plt.plot(hist.history['val_loss'], label='val')
plt.xlabel('Época'); plt.ylabel('MAE')
plt.title(f'Convergencia — mejor modelo V_out={V_OUT}')
plt.legend(); plt.tight_layout(); plt.show()

## 2. Predicción de retornos para 2025
Usamos los últimos V_IN días de 2024 como ventana de entrada para predecir los próximos 90 días.

In [ ]:
# Última ventana disponible de datos históricos (los V_IN días finales de 2024)
last_window = returns_hist.values[-V_IN:].reshape(1, V_IN, 23)   # shape (1, V_IN, 23)
y_pred = model.predict(last_window, verbose=0).flatten()         # retornos predichos por activo

print('Retornos predichos (90 días promedio) por activo:')
for ticker, ret in zip(precios_hist.columns, y_pred):
    print(f'  {ticker:5s}  {ret:+.5f}')

## 3. Construcción de carteras
- **Buy & Hold**: pesos iguales (1/23 por activo)
- **NN**: long proporcional si predicción > 0, short proporcional si < 0
  Normalización: suma de valores absolutos = 1 (presupuesto completo)

In [ ]:
# Cartera 1: Buy & Hold
pesos_bh = np.ones(23) / 23

# Cartera 2: NN (long positivos / short negativos)
pesos_nn = y_pred / np.sum(np.abs(y_pred))   # normalizado a presupuesto = 1

# Visualización de pesos
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
tickers_arr = np.array(precios_hist.columns)
for ax, pesos, titulo in zip(axes,
                              [pesos_bh, pesos_nn],
                              ['Buy & Hold', 'Cartera NN']):
    colors = ['steelblue' if p >= 0 else 'tomato' for p in pesos]
    ax.bar(tickers_arr, pesos, color=colors, edgecolor='k')
    ax.axhline(0, color='k', linewidth=0.8)
    ax.set_title(titulo); ax.set_ylabel('Peso')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
plt.suptitle('Pesos de carteras (fijos durante 2025)', fontsize=12)
plt.tight_layout(); plt.show()

# [EXTENDER] Rebalanceo mensual:
# Cada ~21 días de trading de 2025, tomar la nueva ventana de V_IN días,
# predecir y recalcular pesos_nn.

## 4. Datos de 2025 y cálculo de retornos de carteras

In [ ]:
precios_2025 = yf.download(TICKERS, start='2025-01-01', auto_adjust=True, progress=False)['Close']
# Mantener solo los activos presentes en el modelo
cols_comunes = [t for t in precios_hist.columns if t in precios_2025.columns]
precios_2025 = precios_2025[cols_comunes]
returns_2025 = np.log(precios_2025).diff().dropna()
print(f'Días de trading en 2025: {len(returns_2025)}')

In [ ]:
# Retorno diario de cada cartera = sum(peso_i * retorno_i)
ret_bh = returns_2025.values @ pesos_bh[:len(cols_comunes)]
ret_nn = returns_2025.values @ pesos_nn[:len(cols_comunes)]

# Retorno acumulado
cum_bh = np.exp(np.cumsum(ret_bh)) - 1
cum_nn = np.exp(np.cumsum(ret_nn)) - 1

plt.figure(figsize=(10, 4))
plt.plot(returns_2025.index, cum_bh * 100, label='Buy & Hold', color='steelblue')
plt.plot(returns_2025.index, cum_nn * 100, label='Cartera NN',  color='tomato')
plt.axhline(0, color='k', linewidth=0.8, linestyle='--')
plt.xlabel('Fecha'); plt.ylabel('Retorno acumulado (%)')
plt.title('Comparación de carteras — 2025')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 5. Tabla de métricas comparativa

In [ ]:
TRADING_DAYS = 252

def portfolio_metrics(daily_returns, name):
    """Calcula métricas estándar de cartera sobre retornos diarios log."""
    r = daily_returns
    ret_total      = float(np.exp(np.sum(r)) - 1)
    ret_anual      = float(np.exp(np.mean(r) * TRADING_DAYS) - 1)
    vol_anual      = float(np.std(r) * np.sqrt(TRADING_DAYS))
    sharpe         = ret_anual / vol_anual if vol_anual > 0 else np.nan
    downside       = r[r < 0]
    sortino        = ret_anual / (np.std(downside) * np.sqrt(TRADING_DAYS)) if len(downside) > 0 else np.nan
    cum            = np.exp(np.cumsum(r))
    rolling_max    = np.maximum.accumulate(cum)
    drawdown       = (cum - rolling_max) / rolling_max
    max_drawdown   = float(drawdown.min())

    return {'Retorno total (%)':      round(ret_total  * 100, 2),
            'Retorno anual (%)':      round(ret_anual  * 100, 2),
            'Volatilidad anual (%)':  round(vol_anual  * 100, 2),
            'Sharpe ratio':           round(sharpe,            3),
            'Sortino ratio':          round(sortino,           3),
            'Max Drawdown (%)':       round(max_drawdown * 100, 2)}

m_bh = portfolio_metrics(ret_bh, 'Buy & Hold')
m_nn = portfolio_metrics(ret_nn, 'Cartera NN')

df_metricas = pd.DataFrame([m_bh, m_nn], index=['Buy & Hold', 'Cartera NN']).T
print('\n── Métricas de carteras 2025 ──')
display(df_metricas)